# Structural Signal Validation – secfsn_plus_structural

This notebook validates the structural scoring model using real SEC data.

The pipeline layers are:

- `secfsn_plus` → canonical features
- `secfsn_plus_structural` → structural features + scoring

The purpose is to validate signal quality by checking diagnostics, score distributions, latest-snapshot leaders and laggards, and cross-feature relationships.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
from secfsn_plus import build_feature_panel
from secfsn_plus_structural import build_structural_features


In [ ]:
PERIODS = [(2023, 4), (2024, 1), (2024, 2)]


In [ ]:
panel = build_feature_panel(periods=PERIODS)
features = build_structural_features(panel)

print("Input columns:", len(panel.columns))
print("Output columns:", len(features.columns))


In [ ]:
diagnostics = features.attrs.get("diagnostics", {})
print(diagnostics)


In [ ]:
features.groupby("period")["structural_score"].describe()


In [ ]:
latest = features.index.get_level_values("period").max()
df_latest = features.xs(latest, level="period")

top = df_latest.sort_values("structural_rank").tail(10)
bottom = df_latest.sort_values("structural_rank").head(10)

cols = [
    "structural_score",
    "accrual_intensity",
    "non_cash_earnings_ratio",
    "roe_stability_8q",
    "margin_volatility",
    "signal_decay_autocorr_12q",
]

display(top[cols])
display(bottom[cols])


In [ ]:
features["structural_rank"].hist(bins=50)
plt.title("Structural Rank Distribution")


In [ ]:
cols = [
    "structural_score",
    "accrual_intensity",
    "non_cash_earnings_ratio",
    "roe_stability_8q",
    "margin_volatility",
]

features[cols].corr()


## Interpretation

A high structural score means the model views a company as having stronger structural quality, such as more stable profitability, lower accrual pressure, and earnings that are better supported by cash flow dynamics.

A low structural score means the company appears weaker on those dimensions, which can indicate noisier economics, more fragile earnings quality, or more unstable operating behavior.

In research and trading, this signal can be used as a ranking input, a portfolio screen, or a diagnostic overlay alongside valuation, momentum, or model-based forecasts to separate stronger-quality names from weaker-quality names.
